In [6]:
# ============================================================
# CELL PURPOSE: Import libraries and load datasets
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

DATA_PATH = "/Users/sanjana/Desktop/Hype-Predictor/Data"

# Load datasets
trends = pd.read_csv(f"{DATA_PATH}/google_trends.csv")
reddit = pd.read_csv(f"{DATA_PATH}/reddit_data.csv")
youtube = pd.read_csv(f"{DATA_PATH}/youtube_data.csv")
news = pd.read_csv(f"{DATA_PATH}/news_data.csv")

print("Loaded all datasets")

Loaded all datasets


In [8]:
# ============================================================
# Improved keyword filtering (less strict)
# ============================================================

keywords = {
    "iPhone 17": ["iphone"],
    "PS5 Pro": ["ps5", "playstation"],
    "Nvidia RTX 5090": ["rtx", "nvidia", "gpu"],
    "Air Jordan 11": ["jordan", "nike", "sneakers"],
    "Owala FreeSip": ["owala", "bottle"]
}

def is_relevant(row):
    title = row["title"].lower()
    product = row["product"]
    
    return any(k in title for k in keywords.get(product, []))

reddit_clean = reddit[reddit.apply(is_relevant, axis=1)]

print("After improved keyword filter:", reddit_clean.shape)

After improved keyword filter: (154, 6)


In [10]:
# ============================================================
# Subreddit filtering (balanced approach)
# ============================================================

valid_subreddits = [
    "iphone", "apple",
    "ps5", "playstation", "gaming",
    "nvidia", "buildapc", "pcmasterrace",
    "nike", "sneakers",
    "hydrohomies", "waterbottle"
]

reddit_clean = reddit_clean[
    reddit_clean["subreddit"].str.lower().isin(valid_subreddits)
]

print("After subreddit filter:", reddit_clean.shape)

After subreddit filter: (124, 6)


In [12]:
# ============================================================
# Engagement filtering (final cleanup)
# ============================================================

reddit_clean = reddit_clean[
    (reddit_clean["score"] > 20) &
    (reddit_clean["comments"] > 5)
]

print("After engagement filter:", reddit_clean.shape)

After engagement filter: (54, 6)


In [14]:
# ============================================================
# CELL PURPOSE: Create cleaned Reddit score
# ============================================================

reddit_clean["reddit_score"] = (
    reddit_clean["score"] +
    reddit_clean["comments"]
)

print(reddit_clean.head())

         product    subreddit  \
2  Owala FreeSip  HydroHomies   
3        PS5 Pro          PS5   
4      iPhone 17        apple   
5      iPhone 17       iphone   
7        PS5 Pro  playstation   

                                               title  score  comments  \
2             My new emotional support water bottle!   3599       181   
3  Starfield PS5 players demand refunds, reportin...   3242       631   
4  FBI Extracts Suspect’s Deleted Signal Messages...   2755       390   
5  This is the iPhone roadmap according to analys...   1928       329   
7  Whose still playing on PS3 and PS4 in 2026 due...   1559       615   

   upvote_ratio  reddit_score  
2          0.93          3780  
3          0.87          3873  
4          0.98          3145  
5          0.94          2257  
7          0.91          2174  


In [16]:
# ============================================================
# CELL PURPOSE: Aggregate cleaned Reddit data
# ============================================================

reddit_final = reddit_clean.groupby("product").agg({
    "score": "sum",
    "comments": "sum",
    "reddit_score": "sum"
}).reset_index()

print(reddit_final)

           product  score  comments  reddit_score
0    Air Jordan 11     34        10            44
1  Nvidia RTX 5090   3243      1569          4812
2    Owala FreeSip   3599       181          3780
3          PS5 Pro  12904     28255         41159
4        iPhone 17  10282      2532         12814


In [18]:
#After cleaning, Reddit signals are significantly reduced and more balanced, though some products still exhibit higher engagement, 
#reflecting genuine user interest rather than noise.”

In [20]:
# ============================================================
# CELL PURPOSE: Clean YouTube data (recency filter)
# ============================================================

# Convert to datetime
youtube["published_at"] = pd.to_datetime(youtube["published_at"])

# Keep recent videos (last 2 years)
youtube_clean = youtube[
    youtube["published_at"] >= "2023-01-01"
]

print("Original:", youtube.shape)
print("After date filter:", youtube_clean.shape)

Original: (50, 6)
After date filter: (48, 6)


In [22]:
# ============================================================
# CELL PURPOSE: Aggregate cleaned YouTube data
# ============================================================

youtube_final = youtube_clean.groupby("product").agg({
    "view_count": "sum",
    "like_count": "sum",
    "comment_count": "sum"
}).reset_index()

youtube_final["youtube_score"] = (
    youtube_final["view_count"] +
    youtube_final["like_count"] +
    youtube_final["comment_count"]
)

print(youtube_final)

           product  view_count  like_count  comment_count  youtube_score
0    Air Jordan 11     2741654      109805           2899        2854358
1  Nvidia RTX 5090    14985368      526589          30091       15542048
2    Owala FreeSip     1699868       21079           1277        1722224
3          PS5 Pro     3563703       64684          14669        3643056
4        iPhone 17     8444558      221355           8938        8674851


In [24]:
# ============================================================
# CELL PURPOSE: Prepare Trends data
# ============================================================

trends_long = trends.melt(
    id_vars=["date"],
    var_name="product",
    value_name="interest"
)

trends_final = trends_long.groupby("product").agg({
    "interest": "mean"
}).reset_index()

trends_final.rename(columns={"interest": "trends_score"}, inplace=True)

print(trends_final)

           product  trends_score
0    Air Jordan 11     33.114504
1  Nvidia RTX 5090      9.385496
2    Owala FreeSip     11.656489
3          PS5 Pro     10.225191
4        iPhone 17      4.362595


In [26]:
# ============================================================
# CELL PURPOSE: Prepare News data
# ============================================================

news_final = news[["product", "article_count"]].copy()
news_final.rename(columns={"article_count": "news_score"}, inplace=True)

print(news_final)

           product  news_score
0          PS5 Pro          98
1        iPhone 17          99
2    Air Jordan 11          96
3    Owala FreeSip          27
4  Nvidia RTX 5090         100


In [28]:
# ============================================================
# CELL PURPOSE: Merge all cleaned datasets
# ============================================================

df_clean = trends_final.merge(reddit_final, on="product", how="outer")
df_clean = df_clean.merge(youtube_final, on="product", how="outer")
df_clean = df_clean.merge(news_final, on="product", how="outer")

df_clean.fillna(0, inplace=True)

print(df_clean)

           product  trends_score  score  comments  reddit_score  view_count  \
0    Air Jordan 11     33.114504     34        10            44     2741654   
1  Nvidia RTX 5090      9.385496   3243      1569          4812    14985368   
2    Owala FreeSip     11.656489   3599       181          3780     1699868   
3          PS5 Pro     10.225191  12904     28255         41159     3563703   
4        iPhone 17      4.362595  10282      2532         12814     8444558   

   like_count  comment_count  youtube_score  news_score  
0      109805           2899        2854358          96  
1      526589          30091       15542048         100  
2       21079           1277        1722224          27  
3       64684          14669        3643056          98  
4      221355           8938        8674851          99  


In [30]:
#After cleaning, the dataset shows more balanced and reliable signals, with reduced noise from Reddit and 
#consistent representation across sources.”

In [34]:
df_clean = df_clean[
    ["product", "trends_score", "reddit_score", "youtube_score", "news_score"]
]

In [36]:
df_clean

,product,trends_score,reddit_score,youtube_score,news_score
0,Air Jordan 11,33.114504,44,2854358,96
1,Nvidia RTX 5090,9.385496,4812,15542048,100
2,Owala FreeSip,11.656489,3780,1722224,27
3,PS5 Pro,10.225191,41159,3643056,98
4,iPhone 17,4.362595,12814,8674851,99


In [38]:
df_clean.to_csv(f"{DATA_PATH}/clean_data.csv", index=False)